In [ ]:
import numpy as np

In [ ]:
# ==============================================================================
# VISUALIZATION: STACKED & MASS-BINNED SPECTRA (TNG50-4)
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# --- 0. STYLE CONFIGURATION ---
def matplotlib_default_config():
    font = {'family' : 'serif', 'weight' : 'normal', 'size' : 14}
    rcnew = {
        "mathtext.fontset" : "cm", "text.usetex": False, 'figure.frameon': True,
        'axes.linewidth': 2., "axes.titlesize" : 20, "axes.labelsize" : 20,
        "legend.fontsize" : 14, 'legend.fancybox': True, 'lines.linewidth': 2.5,
        'xtick.alignment': 'center', 'xtick.bottom': True, 'xtick.color': 'black',
        'xtick.direction': 'in', 'xtick.labelbottom': True, 'xtick.labelsize': 18,
        'xtick.major.bottom': True, 'xtick.major.pad': 6.0, 'xtick.major.size': 10.0,
        'xtick.major.width': 1.5, 'xtick.minor.bottom': True, 'xtick.minor.pad': 3.4,
        'xtick.minor.size': 5.0, 'xtick.minor.visible': True, 'xtick.minor.width': 1.0,
        'xtick.top': True, 'ytick.alignment': 'center_baseline', 'ytick.color': 'black',
        'ytick.direction': 'in', 'ytick.labelleft': True, 'ytick.labelsize': 18,
        'ytick.left': True, 'ytick.major.left': True, 'ytick.major.pad': 6.0,
        'ytick.major.size': 10.0, 'ytick.major.width': 1.5, 'ytick.minor.left': True,
        'ytick.minor.pad': 3.4, 'ytick.minor.right': True, 'ytick.minor.size': 5.0,
        'ytick.minor.visible': True, 'ytick.minor.width': 1.0, 'ytick.right': True
    }
    return font, rcnew

# Apply Config
%matplotlib widget 
font, rcnew = matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)

In [ ]:
# --- 1. LOAD TNG DATA ---
# Updated path per your instruction
file_path = '/home/vroberto/FakeSpectra_DLAs/notebooks_TNG/data_for_variance_50_4_mask/TNG50-4_z3_637Halos_DLAMode_Stratified_run0_seed42.pkl'
print("Loading TNG DataFrame...")
df = pd.read_pickle(file_path)

# --- 2. DATA PREPARATION (PHYSICAL UNITS) ---
H_HUBBLE = 0.6774 # TNG Cosmology
# Convert from 10^10 M_sun/h to pure M_sun
df['halo_mass_msun'] = df['halo_mass'] * (1e10 / H_HUBBLE)
df['log_mass_msun'] = np.log10(df['halo_mass_msun'])

# Round impact parameters for clean grouping
df['r_mult_round'] = df['r_multiplier'].round(2)
unique_radii = np.sort(df['r_mult_round'].unique())

print(f"Total Unique Halos in Dataset: {df['halo_id'].nunique()}")
print(f"Mass Range in Dataset: {df['log_mass_msun'].min():.2f} to {df['log_mass_msun'].max():.2f} log10(Msun)")
print(f"Impact Parameters (b/R200): {unique_radii}")

In [ ]:
# ==============================================================================
# PART A: MASS-BINNED COLORED SPECTRA (EQUAL-COUNT BINS)
# ==============================================================================
print("\nGenerating Mass-Binned Colored Spectra...")
NUM_BINS = 4  

# Extract unique halos to avoid biasing the percentiles with multiple skewers per halo
unique_halos = df[['halo_id', 'log_mass_msun']].drop_duplicates()

# Dynamic Quantile Binning
percentiles = np.linspace(0, 100, NUM_BINS + 1)
mass_edges = np.percentile(unique_halos['log_mass_msun'], percentiles)

# Ensure absolute min and max are inclusive
mass_edges[0] -= 0.1
mass_edges[-1] += 0.1

min_mass = mass_edges[0]
max_mass = mass_edges[-1]

print("\n--- Halo Count per Quantile Bin ---")
for i in range(NUM_BINS):
    m_min = mass_edges[i]
    m_max = mass_edges[i+1]
    mask_global = (unique_halos['log_mass_msun'] > m_min) & (unique_halos['log_mass_msun'] <= m_max)
    print(f"Bin {i+1} ({m_min:.2f} to {m_max:.2f}): {unique_halos[mask_global]['halo_id'].nunique()} unique halos")
print("-----------------------------------\n")

# Colormap Configuration
cmap = plt.colormaps['plasma']
norm = mcolors.Normalize(vmin=min_mass, vmax=max_mass)

# Plot Setup
fig_colored, axes_colored = plt.subplots(
    len(unique_radii), 1, 
    figsize=(12, 4.5 * len(unique_radii)), 
    sharex=True
)
if len(unique_radii) == 1: axes_colored = [axes_colored]

lam_rest = 1215.67
# Wavelength grid for interpolation
wave_grid = np.linspace(lam_rest - 10, lam_rest + 10, 500)

for ax_idx, r_target in enumerate(unique_radii):
    ax = axes_colored[ax_idx]
    df_radius = df[df['r_mult_round'] == r_target]
    
    ax.set_title(rf"Impact Parameter: $b \approx {r_target:.2f} R_{{200}}$", pad=10)

    for i in range(NUM_BINS):
        m_min = mass_edges[i]
        m_max = mass_edges[i+1]
        
        mid_mass = (m_min + m_max) / 2.0
        color = cmap(norm(mid_mass))

        df_bin = df_radius[(df_radius['log_mass_msun'] > m_min) & 
                           (df_radius['log_mass_msun'] <= m_max)]
        if df_bin.empty:
            continue

        flux_matrix = []
        
        # Plot individual lines (Raw Data) - Very transparent
        for _, row in df_bin.iterrows():
            w = row['wave']
            f = row['flux']
            
            f_interp = np.interp(wave_grid, w, f)
            flux_matrix.append(f_interp)
            
            ax.plot(wave_grid, f_interp, color=color, alpha=0.03, lw=0.5, 
                    zorder=1, rasterized=True)

        # Calculate and plot the mean line for this mass bin
        if flux_matrix:
            flux_matrix = np.array(flux_matrix)
            mean_flux = np.nanmean(flux_matrix, axis=0)
            ax.plot(wave_grid, mean_flux, color=color, lw=3.0, alpha=0.9, zorder=10)

    # Aesthetics
    ax.axhline(1.0, color='black', linestyle=':', lw=1.5, alpha=0.6, zorder=0)
    ax.axhline(0.0, color='black', linestyle=':', lw=1.5, alpha=0.6, zorder=0)
    ax.axvline(lam_rest, color='gray', linestyle='-.', lw=1.0, alpha=0.8, zorder=0)

    ax.set_ylim(-0.05, 1.1)
    ax.set_ylabel(r'$F \,/\, F_0$')
    # Focus on the absorption core
    ax.set_xlim(lam_rest - 6, lam_rest + 6) 

# X-Axis formatting
axes_colored[-1].set_xlabel(r'Wavelength $\lambda\ [\mathrm{\AA}]$')
axes_colored[-1].axvline(lam_rest, color='gray', linestyle='-.', alpha=0.8, label=r'Ly$\alpha$ Rest Frame')
axes_colored[-1].legend(loc='lower left', frameon=False, fontsize=14)

# Global Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

fig_colored.subplots_adjust(right=0.88, hspace=0.15) 
cbar_ax = fig_colored.add_axes([0.90, 0.15, 0.02, 0.7]) 
cbar = fig_colored.colorbar(sm, cax=cbar_ax)

cbar.set_label(r'$\log_{10}(M_{200} / M_\odot)$', fontsize=22)
cbar.ax.tick_params(labelsize=18)

plt.show()